In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# -----------------------------
# Basic Conv Block
# -----------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


# -----------------------------
# Edge Detection Network (Mini U-Net)
# -----------------------------
class EdgeNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder
        self.enc1 = ConvBlock(3, 32)
        self.enc2 = ConvBlock(32, 64)
        self.enc3 = ConvBlock(64, 128)

        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = ConvBlock(128, 256)

        # Decoder
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = ConvBlock(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = ConvBlock(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = ConvBlock(64, 32)

        # Output edge map
        self.out = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        # Bottleneck
        b = self.bottleneck(self.pool(e3))

        # Decoder
        d3 = self.up3(b)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return torch.sigmoid(self.out(d1))

In [ ]:
class EdgeLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCELoss()

    def forward(self, pred, target):
        # weighted BCE (edges matter more)
        pos_weight = 5.0
        loss = - (pos_weight * target * torch.log(pred + 1e-6)
                  + (1 - target) * torch.log(1 - pred + 1e-6))
        return loss.mean()

In [ ]:
model = EdgeNet().cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = EdgeLoss()

for epoch in range(20):
    for img, edge_gt in dataloader:

        img = img.cuda()
        edge_gt = edge_gt.cuda()

        pred = model(img)

        loss = criterion(pred, edge_gt)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

In [1]:


def get_loader(root_dir, batch_size=8, img_size=256, shuffle=True):
    dataset = BSDS500EdgeDataset(
        root_dir=root_dir,
        split="train",
        img_size=img_size
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=4,
        pin_memory=True
    )

    return loader

val_dataset = BSDS500EdgeDataset(
    root_dir="BSDS500",
    split="val",
    img_size=256
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("nyu-mll/bsds500", split="train", streaming=True)
for sample in dataset:
    image = sample["image"]
    edges = sample["edges"]

In [ ]:
from torchvision.datasets import VOCSegmentation

dataset = VOCSegmentation(
    root="./data",
    download=True
)